# Lab 09-03 · Carrier Transport: Drift and Diffusion

Companion to chapter [09-03](../../chapters/09-03-drift-and-diffusion.md). Handout: [README.md](README.md).

Transport is built here from simulated carriers: an ensemble with random collisions gives the drift velocity, a random walk gives the diffusion coefficient, and slice-by-slice integration gives the sheet resistance of an implanted layer.

In [1]:
import math
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Q, M0, KB = 1.602176634e-19, 9.1093837015e-31, 1.380649e-23
VT = KB * 300 / Q
mun = lambda N: 92 + 1268 / (1 + (N / 1.3e17) ** 0.91)
mup = lambda N: 48 + 402 / (1 + (N / 6.3e16) ** 0.76)
rng = np.random.default_rng(2026)
print(f"kT/q = {VT*1e3:.2f} mV")

kT/q = 25.85 mV


## Part 1 — Drift from an ensemble with random collisions

Each electron starts at rest after a collision, accelerates at $-q\mathcal E/m^*$, and is scattered again after an exponentially distributed time with mean $\tau$. Average the velocity over time and over electrons.

**🔮 Predict (guided exercise):** if $\tau$ is halved, what happens to the drift velocity, the mobility and (Part 2) the diffusion coefficient?

In [2]:
def drift_velocity(tau, m_rel=0.26, field=1e5, n=400_000):
    t = rng.exponential(tau, n)
    a = -Q * field / (m_rel * M0)
    return np.sum(a * t * t / 2) / np.sum(t)          # m/s, time-weighted

tau = 0.199e-12
for tt in (tau, tau / 2):
    vd = drift_velocity(tt)
    mu = -vd / 1e5 * 1e4                              # cm^2/V s
    print(f"tau = {tt*1e12:.4f} ps: v_d = {vd*100:.4e} cm/s, mobility = {mu:.0f} cm^2/V s")
mu_full = -drift_velocity(tau) / 1e5 * 1e4
assert abs(mu_full - 1345) < 15

tau = 0.1990 ps: v_d = -1.3461e+06 cm/s, mobility = 1346 cm^2/V s
tau = 0.0995 ps: v_d = -6.7050e+05 cm/s, mobility = 671 cm^2/V s


## Part 2 — Diffusion from a random walk

Walkers hop $\pm\lambda$ every $\tau$. Their positions spread with variance $2Dt$.

In [3]:
def walk_D(lam, tau, steps=400, walkers=20_000):
    x = rng.choice([-lam, lam], size=(walkers, steps)).sum(axis=1)
    return np.var(x) / (2 * steps * tau)

lam, tau = 45.5e-7, 0.199e-12                          # cm, s
D1 = walk_D(lam, tau)
D2 = walk_D(lam / 2, tau / 2)                          # same speed, half the free time
print(f"D (1-D walk) = {D1:.1f} cm^2/s; with half the free time: {D2:.1f} cm^2/s")
assert abs(D1 / (lam**2 / (2 * tau)) - 1) < 0.03 and abs(D2 / D1 - 0.5) < 0.04

D (1-D walk) = 52.4 cm^2/s; with half the free time: 26.1 cm^2/s


## Part 3 — Sheet resistance of an implanted layer

A dose $Q_d$ = 10¹⁴ cm⁻² is placed as a half-Gaussian of width $\sigma$ at the surface. Integrate $q\mu(N(x))N(x)$ over depth.

**🔮 Predict:** spreading the same dose twice as deep halves the peak doping. Does the sheet resistance rise, fall or stay the same?

In [4]:
def sheet_resistance(sigma, dose=1e14):
    N0 = 2 * dose / (sigma * math.sqrt(2 * math.pi))
    N = lambda x: N0 * math.exp(-x * x / (2 * sigma * sigma))
    g = quad(lambda x: Q * mun(N(x)) * N(x), 0, 12 * sigma, limit=200)[0]
    return 1 / g, N0

R50, N50 = sheet_resistance(50e-7)
R100, N100 = sheet_resistance(100e-7)
print(f"sigma 50 nm: peak {N50:.3e} cm^-3, R_s = {R50:.1f} ohm/sq")
print(f"sigma 100 nm: peak {N100:.3e} cm^-3, R_s = {R100:.1f} ohm/sq (ratio {R50/R100:.3f})")
assert abs(R50 - 498.7) < 0.1 and abs(R100 - 416.1) < 0.1

sigma 50 nm: peak 1.596e+19 cm^-3, R_s = 498.7 ohm/sq
sigma 100 nm: peak 7.979e+18 cm^-3, R_s = 416.1 ohm/sq (ratio 1.199)


## Part 4 — Velocity against field and transit times

In [5]:
def vfit(F, vs, Fc, b): return vs * (F / Fc) / (1 + (F / Fc) ** b) ** (1 / b)
T = 300.0
ve = lambda F: vfit(F, 1.53e9 * T**-0.87, 1.01 * T**1.55, 2.57e-2 * T**0.66)
vh = lambda F: vfit(F, 1.62e8 * T**-0.52, 1.24 * T**1.68, 0.46 * T**0.17)
for L_nm in (1000, 100, 30):
    F = 1.0 / (L_nm * 1e-7)
    print(f"1 V over {L_nm} nm: electrons {L_nm*1e-7/ve(F)*1e12:.3f} ps, holes {L_nm*1e-7/vh(F)*1e12:.3f} ps")
assert abs(ve(1e5) / (1.53e9 * T**-0.87) - 0.955) < 0.002

1 V over 1000 nm: electrons 14.845 ps, holes 29.956 ps
1 V over 100 nm: electrons 0.978 ps, holes 1.320 ps
1 V over 30 nm: electrons 0.284 ps, holes 0.368 ps


## Part 5 — The built-in field of a graded profile

Impose a flat Fermi level on an exponential donor profile, compute the band edge, differentiate it numerically, and check that drift and diffusion currents cancel at every point.

In [6]:
x = np.linspace(0, 1e-4, 10001)
ND = 1e17 * np.exp(-x / 0.5e-4)
Ec = VT * np.log(3.2e19 / ND)                         # E_C - E_F in eV
E = np.gradient(Ec, x)                                # V/cm
mu = mun(ND)
J_drift = Q * ND * mu * E
J_diff = Q * VT * mu * np.gradient(ND, x)
print(f"field {E[5000]:.1f} V/cm (expected {VT/0.5e-4:.1f}); max |J_drift + J_diff| = {np.max(np.abs(J_drift + J_diff)[5:-5]):.2e} A/cm^2")
assert abs(E[5000] - VT / 0.5e-4) < 0.01 and np.max(np.abs(J_drift + J_diff)[5:-5]) < 1e-3 * np.max(np.abs(J_drift))

field 517.0 V/cm (expected 517.0); max |J_drift + J_diff| = 4.42e-05 A/cm^2


## Wrap-up

- Random collisions with mean time $\tau$ turn a constant acceleration into a steady drift: $\mu = q\tau/m^*$.
- A random walk gives Fick's law; with the same thermal speed, halving the free time halves both $\mu$ and $D$, keeping $D/\mu = kT/q$.
- Spreading an implant deeper lowers its sheet resistance at fixed dose, because lightly doped carriers are more mobile.
- In equilibrium, a graded profile builds exactly the field that cancels diffusion.